# Kaggriculture submission builder

Minimal notebook: load `agent/main.py`, run the same static checks `tools/static_check.py` runs, optionally smoke-test it against the real kaggriculture engine's `starter` bot if this notebook is running somewhere that engine is installed, then package `submissions/submission.tar.gz`.

This intentionally does not re-implement the candidate-generation / multi-stage seed-tournament pipeline from the original analysis notebook -- `agent/main.py` here is already the reviewed, checked-in candidate (see `../prompts/reviewer.md` for how it got promoted). This notebook's only job is: verify it, package it.

In [ ]:
from pathlib import Path
import ast, gzip, hashlib, io, json, tarfile

LAB_ROOT = Path.cwd()
if not (LAB_ROOT / "agent" / "main.py").exists():
    LAB_ROOT = LAB_ROOT.parent
assert (LAB_ROOT / "agent" / "main.py").exists(), "run from kaggriculture_agent_lab/ or kaggriculture_agent_lab/notebooks/"

AGENT_PATH = LAB_ROOT / "agent" / "main.py"
SUBMISSION_PATH = LAB_ROOT / "submissions" / "submission.tar.gz"
print(AGENT_PATH)


## 1. Static checks

In [ ]:
source = AGENT_PATH.read_text(encoding="utf-8")
tree = ast.parse(source)
agent_defs = sum(
    1 for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == "agent"
)
assert agent_defs == 1, f"expected exactly one top-level agent(), found {agent_defs}"

namespace = {}
exec(compile(source, str(AGENT_PATH), "exec"), namespace)
assert callable(namespace["agent"])
assert len(namespace["TRACE_ACTIONS"]) == 720

print({"syntax": "PASS", "agent_def_count": agent_defs, "module_load": "PASS"})


## 2. Optional: smoke test against the real engine

Only runs if `kaggle_environments` (with the `kaggriculture` environment) is importable here -- e.g. on an actual Kaggle Notebook for this competition. Safe to skip elsewhere.

In [ ]:
try:
    from kaggle_environments import make
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 12001}, debug=False)
    env.run([str(AGENT_PATH), "starter"])
    final = env.steps[-1]
    print({"left_status": str(final[0].status), "right_status": str(final[1].status)})
except Exception as exc:
    print({"engine_available": False, "reason": repr(exc)})


## 3. Package submission.tar.gz

In [ ]:
raw = io.BytesIO()
with gzip.GzipFile(fileobj=raw, mode="wb", filename="", mtime=0) as zipped:
    with tarfile.open(fileobj=zipped, mode="w") as archive:
        payload = source.encode("utf-8")
        info = tarfile.TarInfo("main.py")
        info.size = len(payload)
        info.mode = 0o644
        info.mtime = 0
        info.uid = info.gid = 0
        info.uname = info.gname = ""
        archive.addfile(info, io.BytesIO(payload))

SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH.write_bytes(raw.getvalue())

with tarfile.open(SUBMISSION_PATH, "r:gz") as archive:
    names = archive.getnames()
    assert names == ["main.py"]
    archived = archive.extractfile("main.py").read().decode("utf-8")
assert archived == source

print({
    "submission": str(SUBMISSION_PATH),
    "submission_bytes": SUBMISSION_PATH.stat().st_size,
    "main_sha256": hashlib.sha256(source.encode("utf-8")).hexdigest(),
    "archive_sha256": hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest(),
})


## Output

Submit `submissions/submission.tar.gz`.